# Transfer Learning from ResNet50

In [1]:
import torch, torchvision
import numpy as np
from collect_data import collect_data, train_model

DATA_DIR = '../data/images/climbs'

# Load pretrained ResNet model
resnet50 = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1)

# Transform images to batch_size x channels x rows x columns
preprocess = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Get training and validation data
train_loader, val_loader = collect_data(DATA_DIR, preprocess)

In [2]:
print(resnet50)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [3]:
class ResNetTransfer(torch.nn.Module):
    def __init__(self, resnetmodel):
        super(ResNetTransfer, self).__init__()
        # ResNet structure (without classifier)
        self.resnet = torch.nn.Sequential(resnet50.conv1,
                                            resnet50.bn1,
                                            resnet50.relu,
                                            resnet50.maxpool,
                                            resnet50.layer1,
                                            resnet50.layer2,
                                            resnet50.layer3,
                                            resnet50.layer4,
                                            resnet50.avgpool)
        # 2048 from the resnet convolution and 1 from the angles
        self.classifier = torch.nn.Sequential(torch.nn.Linear(2049, 23))
    
    def forward(self, X, angles):
        r = self.resnet(X)
        f = torch.flatten(r, 1)
        f = torch.cat([f, angles], dim=1)
        y = self.classifier(f)
        return y

In [4]:
device = torch.device('cpu') 
if torch.cuda.is_available():
    device = torch.device('cuda') # rose laptops and the servers
elif torch.backends.mps.is_available():
    device = torch.device('mps') # mac
print('Using device:', device)

Using device: cuda


In [5]:
import copy

model = ResNetTransfer(copy.deepcopy(resnet50)).to(device)

# Freeze the pretrained convolutional feature extractor. 
# Only the new Linear classifier gets trained.
for param in model.resnet.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params}')

Total trainable parameters: 47150


In [11]:
# Hyperparameters
epsilon = 1E-4

# Use CCE loss
criterion = torch.nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=epsilon)

train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=100, checkpoint_path='resnet_transfer_model.pth')

Epoch [0/100], Train Loss: 2.72481, Val Loss: 2.72844 *
Epoch [10/100], Train Loss: 2.37763, Val Loss: 2.63238
Epoch [20/100], Train Loss: 2.19548, Val Loss: 2.66586
Epoch [30/100], Train Loss: 1.96391, Val Loss: 2.69553
Epoch [40/100], Train Loss: 1.88031, Val Loss: 2.70634
Epoch [50/100], Train Loss: 1.93328, Val Loss: 2.68143
Epoch [60/100], Train Loss: 1.69851, Val Loss: 2.62790 *
Epoch [70/100], Train Loss: 1.92328, Val Loss: 2.64268
Epoch [80/100], Train Loss: 1.63682, Val Loss: 2.71824
Epoch [90/100], Train Loss: 1.64655, Val Loss: 2.74642
Average time per epoch: 0.45s
Best validation loss: 2.62790 (epoch 60)


In [14]:
# Fine-tuning
finetune_model = ResNetTransfer(copy.deepcopy(resnet50)).to(device)
finetune_model.load_state_dict(torch.load('resnet_transfer_model.pth', map_location=device))

for param in finetune_model.resnet.parameters():
    param.requires_grad = True

# Use different learning rate on different params to train resnet layers more slowly
finetune_optimizer = torch.optim.Adam([
    {'params': finetune_model.resnet.parameters(), 'lr': 1e-6},
    {'params': finetune_model.classifier.parameters(), 'lr': epsilon},
])

train_model(finetune_model, train_loader, val_loader, criterion, finetune_optimizer, device, num_epochs=10, checkpoint_path='resnet_finetuned.pth')

Epoch [0/10], Train Loss: 1.73034, Val Loss: 2.63871 *
Epoch [1/10], Train Loss: 1.80482, Val Loss: 2.65673
Epoch [2/10], Train Loss: 1.95603, Val Loss: 2.68471
Epoch [3/10], Train Loss: 1.64402, Val Loss: 2.71678
Epoch [4/10], Train Loss: 1.60268, Val Loss: 2.74689
Epoch [5/10], Train Loss: 1.69211, Val Loss: 2.76950
Epoch [6/10], Train Loss: 1.72512, Val Loss: 2.78113
Epoch [7/10], Train Loss: 1.77440, Val Loss: 2.79091
Epoch [8/10], Train Loss: 1.75343, Val Loss: 2.79207
Epoch [9/10], Train Loss: 1.53982, Val Loss: 2.78641
Average time per epoch: 4.78s
Best validation loss: 2.63871 (epoch 0)
